# Optimizing an ML Pipeline in Azure

Two approaches on the **Bank Marketing** dataset, then a comparison:
1. **Scikit-learn Logistic Regression** with hyperparameters tuned by **HyperDrive**.
2. **AutoML** on the same cleaned data.


## 1. Connect to the workspace and create an experiment

In [ ]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name='udacity-project')

print('Workspace name : ' + ws.name,
      'Azure region   : ' + ws.location,
      'Subscription id : ' + ws.subscription_id,
      'Resource group  : ' + ws.resource_group, sep='\n')

run = exp.start_logging()

## 2. Create (or attach to) a compute cluster

In [ ]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = 'cpu-cluster'

try:
    cpu_cluster = ComputeTarget(workspace=ws, name=cluster_name)
    print('Found existing cluster, using it.')
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(
        vm_size='STANDARD_D2_V2', max_nodes=4)
    cpu_cluster = ComputeTarget.create(ws, cluster_name, compute_config)

cpu_cluster.wait_for_completion(show_output=True)

## 3. HyperDrive configuration

In [ ]:
from azureml.core import ScriptRunConfig, Environment
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import uniform, choice
import os

# Parameter sampler: random search over C and max_iter.
ps = RandomParameterSampling({
    '--C': uniform(0.01, 10.0),
    '--max_iter': choice(50, 100, 150, 200)
})

# Early-termination policy: aggressively stop poorly performing runs.
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1)

# Environment for train.py (uses conda_dependencies.yml in this folder).
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env', file_path='conda_dependencies.yml')

src = ScriptRunConfig(source_directory='.',
                      script='train.py',
                      compute_target=cpu_cluster,
                      environment=sklearn_env)

hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name='Accuracy',
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4)

## 4. Submit the HyperDrive run

In [ ]:
hdr = exp.submit(config=hyperdrive_config)
try:
    from azureml.widgets import RunDetails
    RunDetails(hdr).show()
except Exception:
    pass  # widget unavailable on this kernel; track progress in Studio > Jobs
hdr.wait_for_completion(show_output=True)

## 5. Register the best HyperDrive model

In [ ]:
best_run = hdr.get_best_run_by_primary_metric()
best_run_metrics = best_run.get_metrics()

print('Best run id :', best_run.id)
print('Best run metrics :', best_run_metrics)
print('Best Accuracy :', best_run_metrics['Accuracy'])

best_run.register_model(
    model_name='hyperdrive_best_model',
    model_path='outputs/model.joblib')

## 6. AutoML — prepare the data

Reuse `clean_data` from `train.py` so AutoML trains on exactly the same features.

In [ ]:
# Reuse the exact same loader + cleaning as train.py so AutoML
# trains on identical features. Falls back to pandas if needed.
from train import clean_data, ds, DATA_URL
import pandas as pd

data_src = ds if ds is not None else pd.read_csv(DATA_URL)
x, y = clean_data(data_src)
data = pd.concat([x, y.rename('y')], axis=1)  # AutoML needs the label inside the frame


## 7. AutoML configuration and run

In [ ]:
from azureml.train.automl import AutoMLConfig
from azureml.core import Dataset
import os

# Sanitize column names (AutoML/parquet dislike '.' in names)
data.columns = [c.replace('.', '_') for c in data.columns]

# Save + register a TabularDataset (rubric-friendly, also fine for local run)
os.makedirs('automl_data', exist_ok=True)
data.to_csv('automl_data/bankmarketing_clean.csv', index=False)
datastore = ws.get_default_datastore()
datastore.upload(src_dir='automl_data', target_path='bankmarketing_clean',
                 overwrite=True, show_progress=True)
training_dataset = Dataset.Tabular.from_delimited_files(
    path=[(datastore, 'bankmarketing_clean/bankmarketing_clean.csv')])
training_dataset = training_dataset.register(
    workspace=ws, name='bankmarketing_clean', create_new_version=True)

# Run AutoML LOCALLY (no compute_target) to avoid the broken remote
# curated-environment image build (urllib3 conflict in AutoML-Non-Prod:1.61.0).
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task='classification',
    primary_metric='accuracy',
    training_data=training_dataset,
    label_column_name='y',
    n_cross_validations=5,
    _ignore_package_version_incompatibilities=True)

automl_run = exp.submit(automl_config, show_output=True)
automl_run.wait_for_completion(show_output=True)

## 8. Retrieve and register the best AutoML model

In [ ]:
best_automl_run, fitted_model = automl_run.get_output()
print(best_automl_run)
print(fitted_model)

best_automl_run.register_model(
    model_name='automl_best_model',
    model_path='outputs/')

## 9. Clean up the compute cluster

In [ ]:
cpu_cluster.delete()